# 10-K Parser Comparison: Regex vs XBRL

This notebook compares two approaches for extracting sections from SEC 10-K filings:

1. **Regex on plain text** — strip HTML, find section headers via pattern matching
2. **XBRL structure** — use the inline XBRL tags embedded in the filing HTML

We'll run both on a raw 10-K and compare results. Change `HTML_PATH` below to test different companies.

In [2]:
from bs4 import BeautifulSoup
from pathlib import Path
import re

HTML_PATH = Path("../data/raw/microsoft/10k_latest.html")

with open(HTML_PATH, "rb") as f:
    raw_html = f.read()

print(f"File size: {len(raw_html):,} bytes")

File size: 8,158,067 bytes


---
## Option 1: Regex on Plain Text

The idea: strip all HTML tags to get plain text, then use regex to find section headers like `Item 1.`, `Item 1A.`, etc. Every 10-K uses the same item numbering, so this is company-agnostic.

The challenge is distinguishing **real section headers** from **table-of-contents entries** and **inline references** (e.g. "See Item 1A for details"). We use a heuristic: real headers are followed by substantial text before the next header.

In [3]:
# Step 1: Strip HTML to plain text
soup = BeautifulSoup(raw_html, "html.parser")

for tag in soup(["script", "style", "meta", "noscript"]):
    tag.decompose()

plain_text = soup.get_text(separator="\n", strip=True)
plain_text = re.sub(r"\xa0", " ", plain_text)
plain_text = re.sub(r"\n{3,}", "\n\n", plain_text)

print(f"Plain text length: {len(plain_text):,} chars")
print(f"Preview (first 500 chars):\n")
print(plain_text[:500])

Plain text length: 347,437 chars
Preview (first 500 chars):

10-K
FY
false
0000789019
P2Y
P5Y
P3Y
P1Y
http://fasb.org/us-gaap/2024#DerivativeAssets
http://fasb.org/us-gaap/2024#DerivativeAssets
http://fasb.org/us-gaap/2024#DerivativeLiabilities
http://fasb.org/us-gaap/2024#DerivativeLiabilities
http://fasb.org/us-gaap/2024#ShortTermInvestments
http://fasb.org/us-gaap/2024#ShortTermInvestments
http://fasb.org/us-gaap/2024#OtherAssetsCurrent
http://fasb.org/us-gaap/2024#OtherAssetsCurrent
http://fasb.org/us-gaap/2024#LongTermInvestments
http://fasb.org/us-g


In [4]:
# Step 2: Find ALL occurrences of "Item X." patterns
# Only capture the item number (e.g. "Item 1A"), not the title text after it.
# Titles wrap unpredictably across lines in different filings, so we use the
# standardized item number as the key — it's the same for every 10-K.

ITEM_PATTERN = re.compile(
    r"^\s*(Item\s+\d+[A-C]?)\.",
    re.MULTILINE | re.IGNORECASE
)

all_matches = []
for match in ITEM_PATTERN.finditer(plain_text):
    # Normalize to consistent format: "Item 1A", "Item 7", etc.
    raw_label = match.group(1).strip()
    normalized = re.sub(r"\s+", " ", raw_label).title()

    all_matches.append({
        "label": normalized,
        "position": match.start(),
    })

print(f"Total raw matches: {len(all_matches)}\n")
for m in all_matches:
    print(f"  pos {m['position']:>8,}: {m['label']}")

Total raw matches: 46

  pos   38,186: Item 1
  pos   38,249: Item 1A
  pos   38,274: Item 1B
  pos   38,312: Item 1C
  pos   38,338: Item 2
  pos   38,360: Item 3
  pos   38,389: Item 4
  pos   38,432: Item 5
  pos   38,553: Item 6
  pos   38,575: Item 7
  pos   38,672: Item 7A
  pos   38,743: Item 8
  pos   38,798: Item 9
  pos   38,894: Item 9A
  pos   39,058: Item 9B
  pos   39,088: Item 9C
  pos   39,177: Item 10
  pos   39,245: Item 11
  pos   39,280: Item 12
  pos   39,387: Item 13
  pos   39,473: Item 14
  pos   39,532: Item 15
  pos   39,586: Item 16
  pos   41,419: Item 1
  pos   90,086: Item 1A
  pos  159,054: Item 1B
  pos  159,326: Item 1C
  pos  168,525: Item 2
  pos  169,877: Item 3
  pos  170,079: Item 4
  pos  170,155: Item 5
  pos  172,607: Item 6
  pos  172,645: Item 7
  pos  220,544: Item 7A
  pos  222,484: Item 8
  pos  322,777: Item 9
  pos  322,887: Item 9A
  pos  329,563: Item 9B
  pos  329,927: Item 9C
  pos  330,030: Item 10
  pos  331,640: Item 11
  pos  331,

In [6]:
# Step 3: Filter to real headers using a content-length heuristic
# Real section headers are followed by substantial content (>200 chars) before the next match.
# TOC entries are packed close together with little content between them.

MIN_SECTION_LENGTH = 200

real_headers = []
for i, m in enumerate(all_matches):
    next_pos = all_matches[i + 1]["position"] if i + 1 < len(all_matches) else len(plain_text)
    content_length = next_pos - m["position"]

    if content_length > MIN_SECTION_LENGTH:
        real_headers.append({**m, "content_length": content_length})

print(f"Filtered to {len(real_headers)} real section headers:\n")
for h in real_headers:
    print(f"  {h['label']:<12} ({h['content_length']:,} chars of content)")

Filtered to 20 real section headers:

  Item 16      (1,833 chars of content)
  Item 1       (48,667 chars of content)
  Item 1A      (68,968 chars of content)
  Item 1B      (272 chars of content)
  Item 1C      (9,199 chars of content)
  Item 2       (1,352 chars of content)
  Item 3       (202 chars of content)
  Item 5       (2,452 chars of content)
  Item 7       (47,899 chars of content)
  Item 7A      (1,940 chars of content)
  Item 8       (100,293 chars of content)
  Item 9A      (6,676 chars of content)
  Item 9B      (364 chars of content)
  Item 10      (1,610 chars of content)
  Item 11      (317 chars of content)
  Item 12      (310 chars of content)
  Item 13      (285 chars of content)
  Item 14      (435 chars of content)
  Item 15      (12,862 chars of content)
  Item 16      (1,588 chars of content)


In [7]:
# Step 4: Extract sections and deduplicate
# When the same Item appears twice (e.g. once in a TOC-like area and once as the real section),
# keep the one with more content — that's the actual section body.

raw_sections = {}
for i, h in enumerate(real_headers):
    start = h["position"]
    end = real_headers[i + 1]["position"] if i + 1 < len(real_headers) else len(plain_text)

    # Strip the header line itself, keep only the body content
    content = plain_text[start:end]
    first_newline = content.find("\n")
    if first_newline != -1:
        content = content[first_newline:].strip()

    label = h["label"]
    # Deduplicate: keep whichever occurrence has more content
    if label not in raw_sections or len(content) > len(raw_sections[label]):
        raw_sections[label] = content

# Sort by item number so sections appear in filing order (Item 1, 1A, 1B, ... 16)
def item_sort_key(label):
    match = re.match(r"Item (\d+)([A-C])?", label)
    num = int(match.group(1))
    suffix = match.group(2) or ""
    return (num, suffix)

regex_sections = dict(sorted(raw_sections.items(), key=lambda x: item_sort_key(x[0])))

print(f"Extracted {len(regex_sections)} sections (after dedup):\n")
for name, text in regex_sections.items():
    words = len(text.split())
    print(f"  {name:<12} {words:>6} words")

print(f"\nTotal words: {sum(len(t.split()) for t in regex_sections.values()):,}")

Extracted 19 sections (after dedup):

  Item 1         6905 words
  Item 1A       10116 words
  Item 1B          42 words
  Item 1C        1259 words
  Item 2          198 words
  Item 3           47 words
  Item 5          399 words
  Item 7         7232 words
  Item 7A         273 words
  Item 8        16188 words
  Item 9A         967 words
  Item 9B          72 words
  Item 10         242 words
  Item 11          36 words
  Item 12          33 words
  Item 13          33 words
  Item 14          65 words
  Item 15        1996 words
  Item 16         261 words

Total words: 46,364


In [8]:
# Inspect a section to check quality
sample_key = [k for k in regex_sections if "1A" in k][0]
sample = regex_sections[sample_key]
print(f"=== {sample_key} (first 1000 chars) ===\n")
print(sample[:1000])

=== Item 1A (first 1000 chars) ===

K FACTORS
Our operations and financial results are subject to various risks and uncertainties, including those described below, that could adversely affect our business, operations, financial condition, results of operations, liquidity, and the trading price of our common stock.
STRATEGIC AND COMPETITIVE RISKS
We face intense competition across all markets for our products and services, which could adversely affect our results of operations.
Competition in the technology sector
Our competitors range in size from diversified global companies with significant research and development resources to small, specialized firms whose narrower product lines may let them be more effective in deploying technical, marketing, and financial resources. Barriers to entry in many of our businesses are low and many of the areas in which we compete evolve rapidly with changing and disruptive technologies, shifting user needs, and frequent introductions of new products a

---
## Option 2: XBRL Structure

Modern 10-K filings are submitted as **Inline XBRL (iXBRL)**, where structured data is embedded directly in the HTML using `<ix:nonNumeric>` and `<ix:nonFraction>` tags. These tags carry metadata like `name="us-gaap:RevenueFromContractWithCustomerExcludingAssessedTax"`.

The idea: instead of guessing where sections start from text patterns, use the XBRL tags to identify structured content boundaries.

Let's explore what XBRL data is actually in the filing.

In [7]:
# Step 1: Parse and find all ix: tags
soup_xbrl = BeautifulSoup(raw_html, "html.parser")

# Find all inline XBRL tags (they use the ix: namespace)
# BeautifulSoup lowercases tag names, so ix:nonnumeric, ix:nonfraction, etc.
ix_tags = soup_xbrl.find_all(re.compile(r"^ix:"))

print(f"Total ix: tags found: {len(ix_tags)}\n")

# Count by tag type
from collections import Counter
tag_types = Counter(tag.name for tag in ix_tags)
for tag_type, count in tag_types.most_common():
    print(f"  {tag_type}: {count}")

Total ix: tags found: 1889

  ix:nonfraction: 1551
  ix:nonnumeric: 278
  ix:continuation: 48
  ix:relationship: 4
  ix:footnote: 4
  ix:header: 1
  ix:hidden: 1
  ix:references: 1
  ix:resources: 1


In [8]:
# Step 2: Look at the XBRL namespace attributes — what metadata is available?
# Focus on nonNumeric tags since those contain text content (not financial figures)

non_numeric = soup_xbrl.find_all("ix:nonnumeric")
print(f"ix:nonNumeric tags: {len(non_numeric)}\n")

# What 'name' attributes are used?
name_attrs = Counter(tag.get("name", "no-name") for tag in non_numeric)
print("Top 20 name attributes:")
for name, count in name_attrs.most_common(20):
    print(f"  {name}: {count}")

ix:nonNumeric tags: 278

Top 20 name attributes:
  msft:DebtInstrumentMaturityYear: 21
  us-gaap:AcquiredFiniteLivedIntangibleAssetsWeightedAverageUsefulLife: 14
  msft:DebtInstrumentIssuanceYear: 13
  us-gaap:PropertyPlantAndEquipmentUsefulLife: 9
  us-gaap:DividendsPayableDateDeclaredDayMonthAndYear: 8
  us-gaap:DividendsPayableDateOfRecordDayMonthAndYear: 8
  us-gaap:DividendPayableDateToBePaidDayMonthAndYear: 8
  us-gaap:DerivativeAssetCurrentStatementOfFinancialPositionExtensibleEnumeration: 4
  us-gaap:DerivativeAssetNoncurrentStatementOfFinancialPositionExtensibleEnumeration: 4
  us-gaap:ShareBasedCompensationArrangementByShareBasedPaymentAwardAwardVestingPeriod1: 4
  dei:Security12bTitle: 3
  dei:TradingSymbol: 3
  dei:SecurityExchangeName: 3
  msft:DerivativeAssetStatementOfFinancialPositionExtensibleEnumerationNotDisclosedFlag: 2
  msft:DerivativeLiabilityStatementOfFinancialPositionExtensibleEnumerationNotDisclosedFlag: 2
  us-gaap:DerivativeLiabilityCurrentStatementOfFinanc

In [9]:
# Step 3: Check if there are XBRL tags that map to 10-K sections
# SEC filers sometimes use dei (Document and Entity Information) tags for section structure

dei_tags = [tag for tag in non_numeric if tag.get("name", "").startswith("dei:")]
print(f"DEI tags: {len(dei_tags)}\n")
for tag in dei_tags[:20]:
    text_preview = tag.get_text(strip=True)[:80]
    print(f"  {tag.get('name')}: {text_preview}")

DEI tags: 43

  dei:DocumentFiscalPeriodFocus: FY
  dei:AmendmentFlag: false
  dei:EntityCentralIndexKey: 0000789019
  dei:DocumentType: 10-K
  dei:DocumentAnnualReport: ☒
  dei:DocumentPeriodEndDate: June 30,2025
  dei:CurrentFiscalYearEndDate: June 30
  dei:DocumentFiscalYearFocus: 2025
  dei:DocumentTransitionReport: ☐
  dei:EntityFileNumber: 001-37845
  dei:EntityRegistrantName: MICROSOFT CORPORATION
  dei:EntityIncorporationStateCountryCode: Washington
  dei:EntityTaxIdentificationNumber: 91-1144442
  dei:EntityAddressAddressLine1: ONE MICROSOFT WAY
  dei:EntityAddressCityOrTown: REDMOND
  dei:EntityAddressStateOrProvince: Washington
  dei:EntityAddressPostalZipCode: 98052-6399
  dei:CityAreaCode: 425
  dei:LocalPhoneNumber: 882-8080
  dei:Security12bTitle: Common stock, $0.00000625par value per share


In [10]:
# Step 4: Look for any tags that contain section-level text content
# (i.e., large blocks of text that might correspond to Item sections)

large_text_tags = []
for tag in non_numeric:
    text = tag.get_text(strip=True)
    if len(text) > 500:
        large_text_tags.append({
            "name": tag.get("name", "?"),
            "id": tag.get("id", "?"),
            "length": len(text),
            "preview": text[:100]
        })

print(f"Tags with >500 chars of text: {len(large_text_tags)}\n")
for t in large_text_tags[:15]:
    print(f"  [{t['name']}] (id={t['id']}, {t['length']:,} chars)")
    print(f"    {t['preview']}...\n")

Tags with >500 chars of text: 49

  [cyd:CybersecurityRiskManagementProcessesForAssessingIdentifyingAndManagingThreatsTextBlock] (id=F_cead3a6a-3173-47fe-99a0-50a69674cd49, 4,038 chars)
    ITEM 1C. CYBERSECURITYRISK MANAGEMENT AND STRATEGYMicrosoft plays a central role in the world’s digi...

  [cyd:CybersecurityRiskManagementProcessesIntegratedTextBlock] (id=F_09225809-0ff5-4c42-8843-c7a18071e57c, 1,147 chars)
    We operate a cybersecurity program and governance framework designed to protect our computing enviro...

  [cyd:CybersecurityRiskBoardOfDirectorsOversightTextBlock] (id=F_9dae824f-880f-4ede-8e73-fbcf0ad06a46, 2,445 chars)
    Our Board of Directors oversees cybersecurity risk.Cybersecurity reviews by the Board are scheduled ...

  [cyd:CybersecurityRiskRoleOfManagementTextBlock] (id=F_631c4dc1-e0d1-42ee-affa-9dbeb1ebfe16, 1,319 chars)
    Our CISO leads the strategy, engineering, and operations of cybersecurity across the company, and re...

  [us-gaap:SignificantAccounting

In [11]:
# Step 5: Try to extract sections from XBRL if section-level tags exist
# Look for us-gaap or company-specific tags that wrap entire sections

# Check for common section-wrapping tag patterns
section_candidates = [
    tag for tag in non_numeric
    if any(kw in tag.get("name", "").lower() for kw in [
        "riskfactor", "business", "mda", "property", "legal",
        "cybersecurity", "controls", "compensation"
    ])
]

if section_candidates:
    print(f"Found {len(section_candidates)} section-like XBRL tags:\n")
    for tag in section_candidates:
        text = tag.get_text(strip=True)
        print(f"  {tag.get('name')}: {len(text):,} chars")
else:
    print("No section-wrapping XBRL tags found.")
    print("\nThis means the XBRL data in this filing is mostly for financial figures,")
    print("not for narrative section boundaries. The filing's sections are structured")
    print("only through HTML formatting, not through XBRL markup.")

Found 44 section-like XBRL tags:

  us-gaap:PropertyPlantAndEquipmentUsefulLife: 3 chars
  us-gaap:PropertyPlantAndEquipmentUsefulLife: 3 chars
  us-gaap:PropertyPlantAndEquipmentUsefulLife: 3 chars
  us-gaap:PropertyPlantAndEquipmentUsefulLife: 3 chars
  dei:EntitySmallBusiness: 1 chars
  cyd:CybersecurityRiskManagementProcessesForAssessingIdentifyingAndManagingThreatsTextBlock: 4,038 chars
  cyd:CybersecurityRiskManagementProcessesIntegratedTextBlock: 1,147 chars
  cyd:CybersecurityRiskManagementProcessesIntegratedFlag: 10 chars
  cyd:CybersecurityRiskManagementThirdPartyEngagedFlag: 293 chars
  cyd:CybersecurityRiskThirdPartyOversightAndIdentificationProcessesFlag: 331 chars
  cyd:CybersecurityRiskMateriallyAffectedOrReasonablyLikelyToMateriallyAffectRegistrantFlag: 380 chars
  cyd:CybersecurityRiskBoardOfDirectorsOversightTextBlock: 2,445 chars
  cyd:CybersecurityRiskManagementPositionsOrCommitteesResponsibleReportToBoardFlag: 385 chars
  cyd:CybersecurityRiskProcessForInformingMan

---
## Side-by-Side Comparison

In [ ]:
# Compare what the current style-based parser extracts vs the regex approach
import sys
sys.path.insert(0, "..")
from ingestion.parser import parse_10k

current_sections = parse_10k(HTML_PATH)

print(f"{'Method':<25} {'Sections':>10} {'Total Words':>12}")
print("-" * 50)
print(f"{'Current (style-based)':<25} {len(current_sections):>10} {sum(len(t.split()) for t in current_sections.values()):>12,}")
print(f"{'Regex (plain text)':<25} {len(regex_sections):>10} {sum(len(t.split()) for t in regex_sections.values()):>12,}")

print(f"\n\n--- Sections found by each method ---\n")
print(f"{'Current Parser':<45} {'Regex Parser'}")
print("-" * 70)

current_keys = list(current_sections.keys())
regex_keys = list(regex_sections.keys())
max_len = max(len(current_keys), len(regex_keys))

for i in range(max_len):
    left = current_keys[i][:43] if i < len(current_keys) else ""
    right = regex_keys[i] if i < len(regex_keys) else ""
    print(f"  {left:<45} {right}")

---
## Summary

| | **Option 1: Regex** | **Option 2: XBRL** |
|---|---|---|
| **Company-agnostic?** | Yes — same Item numbering across all 10-Ks | Depends on filer's XBRL tagging |
| **Complexity** | Low — regex + one heuristic | Higher — need to understand XBRL taxonomy |
| **Section detection** | Reliable with content-length filter | May not exist for narrative sections |
| **Financial data** | Gets raw text (messy tables) | Structured values with units/periods |
| **Best for** | Narrative content (Items 1, 1A, 7, etc.) | Extracting specific financial figures |

Run the cells above to see the actual results on Apple's filing, then decide which approach fits your needs.